# 05 — asyncio Event Loops, Coroutines, and Awaitables

asyncio coordinates I/O-oriented concurrency through cooperative suspension. Distinguish coroutine objects, awaited results, tasks, and event-loop lifecycle.

**Level:** Advanced · **Study time:** 90–150 minutes · **Environment:** Python 3.11+ (validated with 3.12); standard library only.

## Learning objectives
- Write and await asynchronous functions
- Explain event-loop scheduling and task creation
- Compare sequential awaits with concurrent gather
- Use asyncio.run appropriately and avoid blocking the event loop

**Method:** Read, predict, execute, vary inputs, and test error paths. The examples avoid live network requests and lengthy sleeps. If an environment has already started an event loop, prefer the standalone asyncio.run examples in a fresh kernel where appropriate.

## Guided lessons

Each section includes theory, runnable code, and a reasoning prompt. Advanced code should minimize hidden state and make cancellation, ownership, and failure behavior explicit.

### 1. Coroutines are not results

Calling an async def function creates a coroutine object; its body executes when awaited or scheduled as a task. asyncio.run is a top-level entry point in ordinary Python scripts, not something to nest inside an already running event loop.

In [ ]:
import asyncio

async def greeting(name):
    await asyncio.sleep(0)
    return f"Hello, {name}"

async def demo():
    return await greeting("Ada")

print(asyncio.run(demo()))

**Think:** How would a Jupyter environment with a running loop differ from an ordinary script?

### 2. Sequential vs concurrent I/O

await suspends one coroutine to let other tasks proceed, but sequential awaits still perform operations in order. asyncio.gather schedules several awaitables together and returns their results in input order.

In [ ]:
import asyncio
async def worker(label):
    await asyncio.sleep(0.001)
    return label.upper()

async def demo():
    serial = [await worker("a"), await worker("b")]
    parallel = await asyncio.gather(worker("a"), worker("b"))
    return serial, parallel

print(asyncio.run(demo()))

**Think:** Why is gather not expected to speed up CPU-bound Python computations?

### 3. Tasks and explicit synchronization

create_task schedules a coroutine on the current loop; tasks may start running before they are awaited. Always retain references to tasks and eventually await them, otherwise failures can be lost or work abandoned.

In [ ]:
import asyncio
async def child(value):
    await asyncio.sleep(0)
    return value * 2

async def parent():
    task = asyncio.create_task(child(21), name="double")
    print("Scheduled:", task.get_name())
    return await task

print(asyncio.run(parent()))

**Think:** What lifecycle problems arise from fire-and-forget tasks?

### 4. Never block the loop with synchronous sleep

time.sleep and blocking network calls halt the thread running the event loop. Use asyncio.sleep for cooperative waiting and asyncio.to_thread for appropriate blocking operations. Do not claim asyncio makes a function intrinsically faster.

In [ ]:
import asyncio
import time

def blocking_lookup(value):
    return value * value

async def safe_demo():
    start = time.perf_counter()
    result = await asyncio.to_thread(blocking_lookup, 12)
    return result, time.perf_counter() - start

value, elapsed = asyncio.run(safe_demo())
print(value, isinstance(elapsed, float))

**Think:** Why is to_thread a boundary for blocking I/O rather than a blanket CPU speedup?

## Exercises

Build a small implementation and test at least one normal and one invalid case. Document timeouts, cancellation rules, or state ownership when relevant.

**Exercise 1.** Implement three async workers and collect results in task-creation order.

In [ ]:
# Exercise 1
# TODO: Write code and tests here.

**Exercise 2.** Compare a series of awaits with gather for artificial non-blocking waits.

In [ ]:
# Exercise 2
# TODO: Write code and tests here.

**Exercise 3.** Write an async function that delegates blocking I/O to asyncio.to_thread.

In [ ]:
# Exercise 3
# TODO: Write code and tests here.

**Exercise 4.** Explain the difference between a coroutine and an asyncio.Task.

In [ ]:
# Exercise 4
# TODO: Write code and tests here.

## Pitfalls and reflection

- Do not call asyncio.run from inside an active event loop.
- Do not create tasks that you never await or otherwise supervise.
- Blocking synchronous operations can stall all cooperative tasks.

**Checkpoint:** Can you explain the failure path and the resource lifetime? What assumption would fail at larger scale?

Restart the kernel and run all cells before moving to the next lesson.